In [0]:

df_spark = spark.read.csv(
    '/Volumes/workspace/testschema/testunitycolume/Mall_Customers.csv',
    header=True,
    inferSchema=True
)
display(df_spark.head(10))

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

# Assemble the three numeric features into a vector
assembler = VectorAssembler(
    inputCols=["Age", "Annual Income (k$)", "Spending Score (1-100)"],
    outputCol="features"
)
df_features = assembler.transform(df_spark)

# Run K-Means clustering
kmeans = KMeans(featuresCol="features", k=4, seed=1)
model = kmeans.fit(df_features)

# View cluster centers
centers = model.clusterCenters()
for idx, center in enumerate(centers):
    print(f"Cluster {idx}: {center}")

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import ClusteringEvaluator

# Assemble features
assembler = VectorAssembler(
    inputCols=["Age", "Annual Income (k$)", "Spending Score (1-100)"],
    outputCol="features"
)
df_features = assembler.transform(df_spark)

# Normalize features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=True, withStd=True)
scaler_model = scaler.fit(df_features)
df_scaled = scaler_model.transform(df_features)

# Experiment with different k and evaluate using silhouette score
scores = []
for k in range(2, 8):
    kmeans = KMeans(featuresCol="scaledFeatures", k=k, seed=1)
    model = kmeans.fit(df_scaled)
    predictions = model.transform(df_scaled)
    evaluator = ClusteringEvaluator(featuresCol="scaledFeatures", metricName="silhouette", distanceMeasure="squaredEuclidean")
    score = evaluator.evaluate(predictions)
    scores.append((k, score))

# Find best k
best_k = max(scores, key=lambda x: x[1])[0]

# Fit final model
kmeans = KMeans(featuresCol="scaledFeatures", k=best_k, seed=1)
model = kmeans.fit(df_scaled)
df_result = model.transform(df_scaled)

# Export to Pandas for visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df_result.select("Age", "Annual Income (k$)", "Spending Score (1-100)", "prediction").toPandas()

# 2D visualization
plt.figure(figsize=(8,6))
sns.scatterplot(data=pdf, x="Annual Income (k$)", y="Spending Score (1-100)", hue="prediction", palette="Set2")
plt.title("Mall Customers Clusters (2D)")
plt.show()

# 3D visualization
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(pdf["Age"], pdf["Annual Income (k$)"], pdf["Spending Score (1-100)"], c=pdf["prediction"], cmap="Set2")
ax.set_xlabel("Age")
ax.set_ylabel("Annual Income (k$)")
ax.set_zlabel("Spending Score (1-100)")
plt.title("Mall Customers Clusters (3D)")
plt.show()